##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Search re-ranking using Gemini embeddings

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Search_reranking_using_embeddings.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

> **Note:** This notebook requires paid tier rate limits to run properly. (cf. [pricing](https://ai.google.dev/pricing) for more details).


This notebook demonstrates the use of embeddings to re-rank search results. This walkthrough will focus on the following objectives:



1.   Setting up your development environment and API access to use Gemini.
2.   Using Gemini's function calling support to access the Wikipedia API.
3.   Embedding content via Gemini API.
4.   Re-ranking the search results.


This is how you will implement search re-ranking:


1.   The user will make a search query.
2.   You will use Wikipedia API to return the relevant search results.
3.   The search results will be embedded and their relevance will be evaluated by calculating distance metrics like cosine similarity.
4.   The most relevant search result will be returned as the final answer.

> The non-source code materials in this notebook are licensed under Creative Commons - Attribution-ShareAlike CC-BY-SA 4.0, https://creativecommons.org/licenses/by-sa/4.0/legalcode.

## Setup

First, download and install the Gemini API Python library.

In [ ]:
%pip install -U -q google-genai

Also install the `wikipedia` package that will be used during this tutorial.

In [ ]:
%pip install -U -q wikipedia

Note: The [`wikipedia` package](https://pypi.org/project/wikipedia/) notes that it was "designed for ease of use and simplicity, not for advanced use", and that production or heavy use should instead "use [Pywikipediabot](http://www.mediawiki.org/wiki/Manual:Pywikipediabot) or one of the other more advanced [Python MediaWiki API wrappers](http://en.wikipedia.org/wiki/Wikipedia:Creating_a_bot#Python)".

In [20]:
import json
import textwrap

from google import genai
from google.genai import types

import wikipedia
from wikipedia.exceptions import DisambiguationError, PageError
wikipedia.set_user_agent("GeminiCookbook/1.0 (https://github.com/google-gemini/cookbook)")

import numpy as np

from IPython.display import Markdown

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

### Grab an API Key

Before you can use the Gemini API, you must first obtain an API key. If you don't already have one, create a key with one click in Google AI Studio.

<a class="button button-primary" href="https://aistudio.google.com/app/apikey" target="_blank" rel="noopener noreferrer">Get an API key</a>

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GEMINI_API_KEY`.

Once you have the API key, pass it to the SDK. You can do this in two ways:

* Put the key in the `GEMINI_API_KEY` environment variable (the SDK will automatically pick it up from there).
* Pass the key to `genai.Client(api_key=...)`

In [ ]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
# Or use `os.getenv('GEMINI_API_KEY')` to fetch an environment variable.
client = genai.Client(api_key=GEMINI_API_KEY)

### Select the model to be used

In [22]:
MODEL_ID = "gemini-3.8-flash" # @param ["gemini-4-argon", "gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}

## Define tools

As stated earlier, this tutorial uses Gemini's function calling support to access the Wikipedia API. Please refer to the [docs](https://ai.google.dev/docs/function_calling) to learn more about function calling.

### Define the search function

To cater to the search engine needs, you will design this function in the following way:


*   For each search query, the search engine will use the `wikipedia.search` method to get relevant topics.
*   From the relevant topics, the engine will choose `n_topics(int)` top candidates and will use `gemini-3.7-flash` to extract relevant information from the page.
*   The engine will avoid duplicate entries by maintaining a search history.


In [23]:
def wikipedia_search(search_queries: list[str]) -> list[str]:
  """Search wikipedia for each query and summarize relevant docs."""
  n_topics=3
  search_history = set() # tracking search history
  search_urls = []
  summary_results = []

  for query in search_queries:
    print(f'Searching for "{query}"')
    search_terms = wikipedia.search(query)

    print(f"Related search terms: {search_terms[:n_topics]}")
    for search_term in search_terms[:n_topics]: # select first `n_topics` candidates
      if search_term in search_history: # check if the topic is already covered
        continue

      print(f'Fetching page: "{search_term}"')
      search_history.add(search_term) # add to search history

      try:
        # extract the relevant data by using `gemini-3.7-flash` model
        page = wikipedia.page(search_term, auto_suggest=False)
        url = page.url
        print(f"Information Source: {url}")
        search_urls.append(url)
        page = page.content
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=textwrap.dedent(f"""\
                Extract relevant information
                about user's query: {query}
                From this source:

                {page}

                Note: Do not summarize. Only Extract and return the relevant information
        """))

        urls = [url]
        if response.candidates[0].citation_metadata:
          extra_citations = response.candidates[0].citation_metadata.citation_sources
          extra_urls = [source.url for source in extra_citations]
          urls.extend(extra_urls)
          search_urls.extend(extra_urls)
          print("Additional citations:", response.candidates[0].citation_metadata.citation_sources)
        try:
          text = response.text
        except ValueError:
          pass
        else:
          summary_results.append(text + "\n\nBased on:\n  " + ',\n  '.join(urls))

      except DisambiguationError:
        print(f"""Results when searching for "{search_term}" (originally for "{query}")
        were ambiguous, hence skipping""")
        continue

      except PageError:
        print(f'{search_term} did not match with any page id, hence skipping.')
        continue
        
      except:
        print(f'{search_term} did not match with any page id, hence skipping.')
        continue

  print(f"Information Sources:")
  for url in search_urls:
    print('    ', url)

  return summary_results


In [24]:
example = wikipedia_search(["What are LLMs?"])

Searching for "What are LLMs?"
Related search terms: ['Large language model', 'List of large language models', 'Retrieval-augmented generation']
Fetching page: "Large language model"
Information Source: https://en.wikipedia.org/wiki/Large_language_model
Large language model did not match with any page id, hence skipping.
Fetching page: "List of large language models"
Information Source: https://en.wikipedia.org/wiki/List_of_large_language_models
List of large language models did not match with any page id, hence skipping.
Fetching page: "Retrieval-augmented generation"
Information Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Information Sources:
     https://en.wikipedia.org/wiki/Large_language_model
     https://en.wikipedia.org/wiki/List_of_large_language_models
     https://en.wikipedia.org/wiki/Retrieval-augmented_generation


Here is what the search results look like:

In [26]:
from IPython.display import display

for e in example:
  display(to_markdown(e))

> Based on the provided text, here is the extracted information regarding what LLMs are, their characteristics, and their limitations:
> 
> * **Definition / Identity:**
>   * LLMs stands for "large language models".
>   * They are referred to as "parametric language model[s]".
>   * Examples include tools like Google's "Bard" (later re-branded to Google Gemini).
> 
> * **How They Operate:**
>   * They have "pre-existing training data" and an "internal representation of [their] training data".
>   * Without retrieval augmentation, they "rely on static training data".
>   * Without prompt stuffing, an "LLM's input is generated by a user".
>   * "In the generative phase, the LLM draws from the augmented prompt and its internal representation of its training data to synthesize" an answer.
> 
> * **Limitations and Behavior:**
>   * "LLMs can provide incorrect information."
>   * They are prone to "hallucinations, which have caused chatbots to describe policies that don't exist, or recommend nonexistent legal cases to lawyers that are looking for citations to support their arguments."
>   * "The LLM can still hallucinate around the source material in its response."
>   * "LLMs can generate misinformation even when pulling from factually correct sources if they misinterpret the context" because an LLM does not "'know' or 'understand' the context."
>   * "LLMs may struggle to recognize when they lack sufficient information to provide a reliable response."
>   * "Without specific training, models may generate answers even when they should indicate uncertainty," as "the model lacks the ability to assess its own knowledge limitations."
> 
> Based on:
>   https://en.wikipedia.org/wiki/Retrieval-augmented_generation

### Pass the tools to the model

If you pass a list of functions to the `GenerativeModel`'s `tools` argument,
it will extract a schema from the function's signature and type hints, and then pass schema along to the API calls. In response the model may return a `FunctionCall` object asking to call the function.

Note: This approach only handles annotations of `AllowedTypes = int | float | str | dict | list['AllowedTypes']`

The request to the Gemini model will keep a reference to the function inself, so that it _can_ execute the function locally later.

## Generate supporting search queries

In order to have multiple supporting search queries to the user's original query, you will ask the model to generate more such queries. This would help the engine to cover the asked question on comprehensive levels.

In [27]:
instructions = """You have access to the Wikipedia API which you will be using
to answer a user's query. Your job is to generate a list of search queries which
might answer a user's question. Be creative by using various key-phrases from
the user's query. To generate variety of queries, ask questions which are
related to  the user's query that might help to find the answer. The more
queries you generate the better are the odds of you finding the correct answer.
Here is an example:

user: Tell me about Cricket World cup 2023 winners.

function_call: wikipedia_search(['What is the name of the team that
won the Cricket World Cup 2023?', 'Who was the captain of the Cricket World Cup
2023 winning team?', 'Which country hosted the Cricket World Cup 2023?', 'What
was the venue of the Cricket World Cup 2023 final match?', 'Cricket World cup 2023',
'Who lifted the Cricket World Cup 2023 trophy?'])

The search function will return a list of article summaries, use these to
answer the  user's question.

Here is the user's query: {query}
"""

## Enable automatic function calling and call the API

Now start a new chat with `enable_automatic_function_calling=True`. With it enabled, the `genai.ChatSession` will handle the back and forth required to call the function, and return the final response:

In [28]:
tools = [wikipedia_search]

config = types.GenerateContentConfig(
    tools=tools
)

In [29]:
chat = client.chats.create(
    model="gemini-3.8-flash",
    config=config
)

query = "Explain how deep-sea life survives."

res = chat.send_message(instructions.format(query=query))

Searching for "Deep sea biology"
Related search terms: ['Deep-sea gigantism', 'Deep sea', 'Ceratioidei']
Fetching page: "Deep-sea gigantism"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_gigantism
Deep-sea gigantism did not match with any page id, hence skipping.
Fetching page: "Deep sea"
Information Source: https://en.wikipedia.org/wiki/Deep_sea
Deep sea did not match with any page id, hence skipping.
Fetching page: "Ceratioidei"
Information Source: https://en.wikipedia.org/wiki/Ceratioidei
Searching for "Deep sea creature adaptations"
Related search terms: ['Deep sea', 'Deep-sea community', 'Deep-sea fish']
Fetching page: "Deep-sea community"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_community
Deep-sea community did not match with any page id, hence skipping.
Fetching page: "Deep-sea fish"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_fish
Searching for "How do organisms survive in the deep sea?"
Related search terms: ['Deep sea', 'Deep-sea 

In [30]:
to_markdown(res.text)

> Deep-sea organisms survive in one of Earth’s most extreme environments—characterized by crushing hydrostatic pressure, near-freezing temperatures, perpetual darkness, and scarce nutrients—through remarkable biochemical, anatomical, physiological, and ecological adaptations:
> 
> ---
> 
> ### 1. Coping with Extreme Hydrostatic Pressure
> At depths thousands of meters down, pressure can exceed several hundred atmospheres. Deep-sea life survives without being crushed through adaptations at both the cellular and structural levels:
> * **Internal Pressure Equalization:** Organisms match their internal fluid pressures to the external hydrostatic pressure, eliminating the pressure differentials that would otherwise crush gas-filled structures.
> * **Absence or Modification of Swim Bladders:** Most deep-sea fish have lost gas-filled swim bladders (which would rupture or require immense energy to maintain) or have replaced the gas with lipids and fats.
> * **Cell Membrane Fluidity:** Extreme pressure compresses molecules and solidifies lipid bilayers. Deep-sea organisms (piezophiles) incorporate higher levels of **unsaturated fatty acids** and branched lipids into their cell membranes to maintain fluidity and function.
> * **Pressure-Resistant Proteins and Enzymes:** Cellular enzymes and structural proteins are adapted with tighter packing, fewer interior void spaces, and modified tertiary bonds, preventing pressure from denaturing them or stalling vital biochemical reactions.
> * **Reduced Skeletal Density:** Many species have minimal, lightweight bones, reduced calcification, and watery, gelatinous flesh to reduce mass, improve buoyancy, and conserve energy.
> 
> ---
> 
> ### 2. Navigating and Thriving in Total Darkness
> Below the photic zone (~200 meters), sunlight fades to total darkness. Life relies on sensory specializations and self-generated light:
> * **Bioluminescence:** Around 75% or more of deep-sea pelagic organisms produce light using luciferin-luciferase reactions or symbiotic bacteria (*Photobacterium*). It is used for:
>   * **Counterillumination:** Photophores on an animal’s underside emit dim light matching downwelling sunlight from above, masking their silhouette from predators below (e.g., lanternfish, hatchetfish, squids).
>   * **Luring Prey:** Structures like the modified dorsal fin (*esca*) in anglerfish act as glowing bait to draw unsuspecting prey close.
>   * **Defense:** Certain squids and shrimp eject blinding glowing clouds or autotomize (shed) flashing limbs to distract predators while they escape.
>   * **Illumination:** Specialized dragonfish emit **red bioluminescence**, acting as an invisible "sniper light" to illuminate prey; most deep-sea creatures cannot see red wavelengths, as their vision is tuned exclusively to blue-green light.
> * **Sensory Enhancements:** Many deep-sea fish possess extremely enlarged or tubular upward-facing eyes, multiplied rhodopsin (*Rh1*) opsin genes, and mirrored retina structures to detect faint flashes. Others rely on extensive lateral line systems and highly developed olfactory senses to locate food and mates.
> 
> ---
> 
> ### 3. Surviving Severe Food Scarcity
> Because photosynthesis cannot occur, primary organic food is limited:
> * **Low Metabolic Rates & Sit-and-Wait Predation:** Many bathypelagic animals drift passively or float motionlessly to conserve energy, expending calories only in short bursts to ambush prey.
> * **Extreme Opportunistic Feeding:** When food appears, deep-sea predators are equipped to consume meals equal to or larger than themselves:
>   * Disproportionately large, gaping mouths with long, recurved teeth.
>   * Hinged, dislocating jaws and elastic, highly distensible stomachs (such as in gulper eels and black swallowers).
>   * Black or heavily pigmented stomach linings that prevent swallowed bioluminescent prey from glowing through their stomachs and alerting larger predators.
> * **Scavenging "Marine Snow":** Detritus, fecal matter, and decaying organic debris (known as marine snow) drift down from surface waters, serving as a vital baseline food source for benthic scavengers like sea cucumbers, amphipods, and crabs.
> 
> ---
> 
> ### 4. Chemosynthesis: An Alternative Energy Source
> Around hydrothermal vents and cold seeps, entire ecosystems flourish independent of sunlight:
> * **Chemolithoautotrophy:** Chemosynthetic bacteria and archaea utilize reduced chemical compounds discharged from vents (such as hydrogen sulfide ($\text{H}_2\text{S}$), methane ($\text{CH}_4$), and molecular hydrogen) to synthesize organic carbon from $\text{CO}_2$.
> * **Symbiotic Relationships:** Animals like giant tube worms (*Riftia pachyptila*) and hydrothermal vent clams have no mouths or digestive tracts; instead, they house millions of chemotrophic bacteria in specialized organs (trophosomes), trading inorganic chemical substrates transported by the host's blood for nutrition synthesized by the bacteria.
> 
> ---
> 
> ### 5. Specialized Reproductive Strategies
> Because deep-sea populations are sparsely distributed across vast, dark ocean expanses, finding mates presents a major challenge:
> * **Bioluminescent Signaling:** Unique flashing rhythms and patterns allow species to recognize compatible mates in the dark.
> * **Pheromone Detection:** Males often possess oversized olfactory organs to detect tiny concentrations of female pheromones across long distances.
> * **Extreme Sexual Parasitism:** In ceratioid anglerfish, miniature males seek out large females and bite into their skin. An enzyme dissolves their tissues, permanently fusing their bloodstreams and organs together. The male's non-reproductive organs degenerate, turning him into a permanent sperm provider nourished directly by the female's circulatory system.
> * **Hermaphroditism:** Many deep-sea invertebrates and fishes are hermaphrodites, ensuring that any encounter between two mature individuals can result in successful fertilization.

That looks like it worked. You can go through the chat history to see the details of what was sent and received in the function calls:

In [31]:
for content in chat._comprehensive_history:
  part = content.parts[0]

  print(f'{content.role} -> ', end='')
  print(json.dumps(part.to_json_dict(), indent=2))
  print('---' * 20)


user -> {
  "text": "You have access to the Wikipedia API which you will be using\nto answer a user's query. Your job is to generate a list of search queries which\nmight answer a user's question. Be creative by using various key-phrases from\nthe user's query. To generate variety of queries, ask questions which are\nrelated to  the user's query that might help to find the answer. The more\nqueries you generate the better are the odds of you finding the correct answer.\nHere is an example:\n\nuser: Tell me about Cricket World cup 2023 winners.\n\nfunction_call: wikipedia_search(['What is the name of the team that\nwon the Cricket World Cup 2023?', 'Who was the captain of the Cricket World Cup\n2023 winning team?', 'Which country hosted the Cricket World Cup 2023?', 'What\nwas the venue of the Cricket World Cup 2023 final match?', 'Cricket World cup 2023',\n'Who lifted the Cricket World Cup 2023 trophy?'])\n\nThe search function will return a list of article summaries, use these to\nans

In the chat history you can see all 4 steps:

1. The user sent the query.
2. The model replied with a `genai.protos.FunctionCall` calling the `wikipedia_search` with a number of relevant searches.
3. Because you set `enable_automatic_function_calling=True` when creating the `genai.ChatSession`, it  executed the search function and returned the list of article summaries to the model.
4. Folliwing the instructions in the prompt, the model generated a final answer based on those summaries.


## [Optional] Manually execute the function call

If you want to understand what happened behind the scenes, this section executes the `FunctionCall` manually to demonstrate.

In [32]:
manual_config = types.GenerateContentConfig(
    tools=tools,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="ANY")
    ),
)
chat = client.chats.create(
    model=MODEL_ID,
    config=manual_config,
)

In [33]:
result = chat.send_message(instructions.format(query=query))

Initially the model returns a FunctionCall:

In [34]:
fc = result.function_calls[0].to_json_dict()
print(json.dumps(fc, indent=2))


{
  "id": "call_596802",
  "args": {
    "search_queries": [
      "Deep-sea biology",
      "Deep-sea fish adaptations",
      "How do organisms survive in the deep sea?",
      "Hydrothermal vent microbial community",
      "Chemosynthesis in the deep ocean",
      "Adaptations to high hydrostatic pressure in marine organisms",
      "Marine snow deep sea food web",
      "Bioluminescence in deep-sea creatures",
      "Piezophile"
    ]
  },
  "name": "wikipedia_search"
}


In [35]:
fc['name']

'wikipedia_search'

Call the function with generated arguments to get the results.

In [36]:
summaries = wikipedia_search(**fc['args'])

Searching for "Deep-sea biology"
Related search terms: ['Deep-sea gigantism', 'Deep sea', 'Deep sea mining']
Fetching page: "Deep-sea gigantism"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_gigantism
Deep-sea gigantism did not match with any page id, hence skipping.
Fetching page: "Deep sea"
Information Source: https://en.wikipedia.org/wiki/Deep_sea
Deep sea did not match with any page id, hence skipping.
Fetching page: "Deep sea mining"
Information Source: https://en.wikipedia.org/wiki/Deep_sea_mining
Deep sea mining did not match with any page id, hence skipping.
Searching for "Deep-sea fish adaptations"
Related search terms: ['Deep-sea fish', 'Deep sea', 'Deep-sea community']
Fetching page: "Deep-sea fish"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_fish
Fetching page: "Deep-sea community"
Information Source: https://en.wikipedia.org/wiki/Deep-sea_community
Deep-sea community did not match with any page id, hence skipping.
Searching for "How do organisms

Now send the `FunctionResult` to the model.

In [54]:
# The function call was executed manually above, so don't force another one:
# tell the model no more tool calls are allowed and let it answer in text.
response = chat.send_message(
    [genai.types.Part.from_function_response(
        name='wikipedia_search',
        response={'result': summaries}
    )],
    config=types.GenerateContentConfig(
        tools=tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode="NONE")
        ),
    ),
)
to_markdown(response.text)


> Deep-sea organisms survive in one of the planet's most extreme environments—characterized by crushing hydrostatic pressure, perpetual darkness, near-freezing temperatures, and severe food scarcity—through specialized cellular, physiological, anatomical, and ecological adaptations:
> 
> ---
> 
> ### 1. Resisting Crushing Hydrostatic Pressure
> At depths exceeding thousands of meters, pressure can reach hundreds to over 1,000 times atmospheric pressure at sea level. Organisms endure this through key biochemical and structural adaptations:
> * **Equal Internal Pressure:** Deep-sea animals do not get crushed because their internal body tissues and fluids exert the same outward pressure as the surrounding water.
> * **Membrane Fluidity:** Extreme pressure tends to compress and solidify lipid bilayers. To keep cell membranes fluid and permeable, deep-sea organisms and pressure-loving microbes (**piezophiles**) incorporate a high proportion of unsaturated and branched fatty acids into their phospholipid membranes.
> * **Pressure-Resistant Proteins:** Hydrostatic pressure inhibits biochemical reactions that increase in volume and can deform protein structures. Deep-sea species utilize structurally rigid, tightly packed enzymes and globular proteins with minimal internal void space, preventing compaction and inactivation under pressure.
> * **Elimination of Gas Cavities:** Gas compresses drastically under high pressure, making air-filled spaces hazardous and metabolically costly. Consequently, most deep-sea fish eliminate or reduce gas-filled swim bladders, relying instead on watery tissues, reduced skeletons, fat deposits, or low-density gelatinous layers for buoyancy.
> 
> ---
> 
> ### 2. Navigating Total Darkness
> Below 200 meters (the twilight zone), sunlight quickly fades, and beyond 1,000 meters, it is entirely absent:
> * **Enhanced Vision:** Many species possess large, hyper-sensitive eyes containing exclusively rod cells and multiplied vision-pigment genes (such as *Rh1* opsins) to detect faint bioluminescence. Others, like spookfish and hatchetfish, have upward-directed tubular eyes or reflective mirror optics to spot prey silhouetted against dim downwelling light.
> * **Alternative Senses:** Blind or small-eyed creatures compensate with elongated tactile feelers, highly developed olfactory systems to detect faint chemical scents, and sensitive lateral line systems to register subtle water vibrations and pressure changes.
> * **Bioluminescence:** Roughly 75% of deep-sea pelagic animals produce light via enzymatic reactions (luciferin oxidized by luciferase) or through symbiotic luminous bacteria:
>   * **Counter-illumination:** Organisms (like lanternfish and squids) emit light from photophores along their bellies to match downwelling ambient light, concealing their silhouette from predators lurking below.
>   * **Aggressive Mimicry:** Anglerfish dangle a bioluminescent lure (*esca*) to attract unsuspecting prey close to their jaws.
>   * **Defense:** Animals such as deep-sea shrimp and squids expel glowing bioluminescent clouds or discard flashing limbs to blind and distract predators during escape.
>   * **Specialized Spotlights:** Dragonfishes can emit red light—invisible to most deep-sea organisms—allowing them to illuminate prey undetected.
> * **Color Camouflage:** Ambush predators often have dark red or jet-black skin. Because water rapidly absorbs red wavelengths, red organisms appear completely black in deep water.
> 
> ---
> 
> ### 3. Coping with Food Scarcity
> Because sunlight cannot penetrate to fuel photosynthesis, most deep-sea life relies on scarce organic debris ("marine snow") drifting down from the surface or rare bonanzas like fallen whale carcasses:
> * **Low Metabolic Rates:** Many bathypelagic animals are sedentary "sit-and-wait" ambush predators that conserve energy between infrequent meals through very slow metabolisms and weak, watery musculature.
> * **Extreme Feeding Anatomy:** To capitalize on any available food, deep-sea predators often feature disproportionately huge, hinged jaws, needle-sharp recurved teeth, and highly distensible stomachs, enabling them to swallow prey equal to or larger than themselves.
> * **Concealing Digestion:** Many predatory deep-sea fish have jet-black stomach linings to block light if they consume prey that is actively bioluminescing, preventing them from revealing their location to larger predators.
> * **Diel Vertical Migration:** Certain mesopelagic animals swim hundreds of meters upward toward surface waters at night to feed under cover of darkness, descending back into the deep before sunrise.
> 
> ---
> 
> ### 4. Alternative Energy via Chemosynthesis
> In unique ecosystems such as hydrothermal vents and cold seeps, life decouples entirely from surface photosynthesis:
> * **Chemolithoautotrophy:** Archaea and bacteria harness chemical energy from mineral-rich fluids (such as hydrogen sulfide, methane, and iron compounds) to synthesize organic matter.
> * **Symbiosis:** Complex organisms, including giant tube worms (*Riftia pachyptila*) and vent mussels, house these chemosynthetic bacteria in specialized tissues, receiving direct nutrition in exchange for transporting vent chemicals to their symbionts.
> 
> ---
> 
> ### 5. Reproductive Strategies in a Vast Void
> Because population density is low and distances are vast, locating a mate is a major challenge:
> * **Pheromones and Light Signals:** Species use unique bioluminescent flashing codes and potent chemical pheromones to signal their presence across large distances.
> * **Hermaphroditism:** Many deep-sea species are simultaneous hermaphrodites, ensuring that any rare encounter between two individuals can result in successful fertilization.
> * **Sexual Parasitism:** In some anglerfish species, tiny males locate a female by scent. The male bites the female, and their tissues and blood vessels permanently fuse. The male degenerates into an attached sperm-producing organ, guaranteeing an immediate mate whenever the female is ready to spawn.

## Re-ranking the search results

Helper function to embed the content:

In [40]:
EMBEDDINGS_MODEL_ID = "gemini-embedding-001"  # @param ["gemini-embedding-2-preview", "gemini-embedding-001"] {"allow-input": true, "isTemplate": true}

def get_embeddings(content: list[str]) -> np.ndarray:
  embeddings = client.models.embed_content(
    model=EMBEDDINGS_MODEL_ID,
    contents=content,
    config=types.EmbedContentConfig(
      task_type='SEMANTIC_SIMILARITY'
    )
  )
  return embeddings.embeddings[0].values

Please refer to the [embeddings guide](https://ai.google.dev/gemini-api/docs/embeddings) for more information on embeddings.

Your next step is to define functions that you can use to calculate similarity scores between two embedding vectors. These scores will help you decide which embedding vector is the most relevant vector to the user's query.


You will now implement cosine similarity as your metric. Here returned embedding vectors will be of unit length and hence their L1 norm (`np.linalg.norm()`) will be ~1. Hence, calculating cosine similarity is esentially same as calculating their dot product score.

In [41]:
def dot_product(a: np.ndarray, b: np.ndarray):
  return (np.array(a) @ np.array(b).T)

### Similarity with user's query

Now it's time to find the most relevant search result returned by the Wikipedia API.

Use Gemini API to get embeddings for user's query and search results.

In [42]:
search_res = get_embeddings(summaries)
embedded_query = get_embeddings([query])

Calculate similarity score:

In [43]:
sim_value = dot_product(search_res, embedded_query)

using `np.argmax` best candidate is selected.

**Users's Input:** Explain how deep-sea life survives.

**Answer:**

In [44]:
Markdown(summaries[np.argmax(sim_value)])

Based on the provided source, here is the relevant information regarding deep-sea fish adaptations:

### Cellular, Biochemical, and Pressure Adaptations
* **Internal Pressure Balance:** Deep-sea organisms have the same pressure within their bodies as is exerted on them from the outside, preventing them from being crushed.
* **Membrane Fluidity:** To counteract reduced membrane fluidity caused by extreme hydrostatic pressure, deep-sea cellular membranes favor phospholipid bilayers with an increased proportion of unsaturated fatty acids.
* **Volume-Decreasing Metabolic Reactions:** High pressure inhibits biochemical reactions that increase in volume and enhances those associated with a decrease in volume; deep-sea fish have evolved metabolic processes and reaction criteria that ultimately decrease system volume.
* **Enzyme and Protein Resistance:** 
  * Deep-sea species exhibit lower changes of entropy and enthalpy, favoring negative enthalpy changes and reduced dependence on entropy-driven reactions.
  * Globular proteins (such as the tertiary structure of G-actin in muscle fibers) are relatively rigid compared to surface fish, structurally strengthened to resist pressure through modification of bonds in their tertiary structure (which also makes them extremely heat-resistant).
  * The lipoprotein enzyme $\text{Na}^+/\text{K}^+$-ATPase features adaptations in its protein and lipid moieties that give it a much higher tolerance to hydrostatic pressure and compression.
* **Gelatinous Layer:** About 200 known species (across 10 orders and 13 families) possess a gelatinous layer below the skin or around the spine for buoyancy, low-cost growth, and reducing drag to increase swimming efficiency.

### Buoyancy and Body Structure Adaptations
* **Lack of Swim Bladders / Hydrofoils:** Because high pressure makes maintaining a gas-filled swim bladder energetically costly or prone to compression/expansion risks, swim bladders in deep-sea fishes (particularly bathypelagic species) are usually absent, withered, filled with fat, or scarcely operational. Instead, many exhibit structures similar to hydrofoils to provide hydrodynamic lift.
  * *(Exceptions: The cusk-eel and rough abyssal grenadier maintain functional swim bladders at extreme depths, allowing them to retain high bone density and strong jaws).*
* **Reduced Density and Minimal Skeletons:** The deeper a fish lives, the more jelly-like its flesh and the more minimal its bone structure. Tissue density is reduced via high fat content, water accumulation, and reduced skeletal size, thickness, and mineral content.
* **High Water Content:** Bodies are watery and elongated with weak muscles; having bodies largely composed of water prevents them from being compressed by deep-sea pressures.
* **Larval and Egg Adaptations:** Eggs and larvae typically drift in the water column and maintain neutral buoyancy using oil droplets in their plasma.
* **Neoteny:** Bathypelagic fish retain larval-like features (slimy skin without scales; reduced gills, kidneys, hearts, and swim bladders; watery muscles and minimal skeletons) through neoteny, allowing them to stay suspended with minimal energy expenditure.
* **Locomotion:** Body shapes are typically adapted for periodic bursts of swimming rather than constant swimming.

### Vision and Sensory Adaptations
* **Large, Sensitive Eyes:** Non-blind deep-sea fish possess large, highly sensitive eyes (up to 100 times more sensitive to light than human eyes) adapted to register faint bioluminescence.
* **Multiple Rh1 Opsin Genes:** While most vertebrates have one Rh1 gene, some deep-sea fish have several, and the silver spinyfin (*Diretmus argenteus*) has 38, aiding vision in dim depths.
* **Tubular Eyes:** Some deep-water mesopelagic fish have upward-directed tubular eyes with large lenses and exclusively rod cells, providing binocular vision to spot prey silhouetted against ambient light above.
* **Mirror Optics:** The brownsnout spookfish uses an internal mirror rather than a lens to focus images.
* **Yellow Lenses:** Some predators possess yellow lenses that filter ambient downwelling light, making prey bioluminescence visible.
* **Alternative Senses / Blindness:** Many species are blind and rely instead on sensitivities to changes in local pressure (via the lateral line), sound (inner ear), or smell (olfactory systems) to detect food and mates. Some have long feelers to navigate and locate prey.

### Bioluminescence and Camouflage
* **Bioluminescence Production:** Many species possess photophores—light-producing glandular cells containing luminous bacteria bordered by dark pigments, sometimes equipped with lenses to regulate light emanation. Producing light consumes only about 1% of the fish's energy.
* **Functions of Bioluminescence:** Used to attract prey (e.g., the anglerfish's illuminated lure), find mates, signal territory, communicate, and distract or temporarily blind predators.
* **Counter-illumination:** Mesopelagic fish illuminate photophores on their bellies to match the color and intensity of downwelling light, eliminating their silhouette from predators lurking below.
* **Color Camouflage:** 
  * Ambush predators are colored dark, black, or red (red light wavelengths do not penetrate deep water, making red appear black).
  * Migratory mesopelagic forms use countershaded silvery coloration.

### Feeding and Predation Adaptations
* **Lures and Feelers:** Deep-sea anglerfish have an illicium (a fishing-rod-like facial adaptation) tipped with a bioluminescent, worm-like piece of skin to lure prey.
* **Coping with Infrequent Meals:** Due to low food availability, fish adapt to consume prey equal to or larger than themselves by developing:
  * Disproportionately large mouths
  * Great, recurved, sharp teeth
  * Hinged, extensible jaws
  * Expandable bodies/stomachs
  * Overlapping gill rakers that keep swallowed prey from escaping.
* **Low-Energy Foraging:** Bathypelagic species are predominantly sedentary, "sit-and-wait" or "lie-in-wait" ambush predators with slow metabolisms and unspecialized diets to conserve energy.

### Vertical Migration and Temperature Adaptations
* **Daily Vertical Migrations:** Many mesopelagic fish migrate up to the epipelagic zone at dusk to feed on plankton and descend at dawn. They use swim bladders (inflating them to ascend and deflating them to descend).
* **Thermal Tolerance:** Migrating through ocean thermoclines exposes mesopelagic fish to temperature shifts between 10 °C and 21 °C (50–69 °F), requiring wide temperature tolerances.

### Reproductive Adaptations
* **Finding Mates in Darkness:** Species use species-specific bioluminescent patterns, release of pheromones, and highly developed olfactory systems (allowing males to track females by scent).
* **Hermaphroditism:** Some species are hermaphrodites, ensuring any encounter between two individuals can result in egg and sperm production.
* **Parasitic Mating (Sexual Dimorphism):** In certain anglerfishes (e.g., *Haplophryne mollis*), a tiny male bites the female and releases an enzyme that dissolves the tissue of his mouth and her skin, permanently fusing their circulatory systems. The male atrophies into a sperm-producing organ attached to the female, ensuring an immediate mate when she is ready to spawn.

Based on:
  https://en.wikipedia.org/wiki/Deep-sea_fish

### Similarity with Hypothetical Document Embeddings (HyDE)

Drawing inspiration from [Gao et al](https://arxiv.org/abs/2212.10496) the objective here is to generate a template answer to the user's query using `gemini-3.7-flash`'s internal knowledge. This hypothetical answer will serve as a baseline to calculate relevance of all the search results.

In [45]:
res = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Generate a hypothetical answer
to the user's query by using your own knowledge. Assume that you know everything
about the said topic. Do not use factual information, instead use placeholders
to complete your answer. Your answer should feel like it has been written by a human.

query: {query}""")

to_markdown(res.text)

> When you look at the conditions down past [Depth Level], it honestly seems impossible that anything could exist, let alone thrive. Survival at that level comes down to mastering three main obstacles: crushing force, the total absence of solar radiation, and extreme cold. 
> 
> Here is how these creatures pull it off:
> 
> ### 1. Counteracting the Pressure
> To prevent being crushed under [Pressure Metric], deep-sea organisms completely abandon standard skeletal and cavity structures. Instead of hollow spaces like lungs, their bodies are permeated with [Fluid Compound A], which creates an equalized hydrostatic balance with the surrounding water. At the cellular level, they produce high concentrations of [Stabilizing Enzyme B] and [Lipid Type C] to keep their cell membranes flexible and functional, preventing their internal architecture from collapsing under [Force Measurement].
> 
> ### 2. Generating Energy in the Dark
> Because solar-driven photosynthesis is impossible past [Sunlight Penetration Limit], the entire food web is powered by [Alternative Bio-Cycle D]. Near [Geological Vent Formations], the crust expels superheated mixtures rich in [Compound X] and [Mineral Y]. Specialized microbes harvest these materials to create energy through [Process Name 1]. Larger fauna, such as [Species Placeholder A] and [Species Placeholder B], survive either by grazing on these bacterial colonies or by housing them within specialized structures called [Symbiotic Organ Z] to feed directly off their chemical output.
> 
> ### 3. Sensory Adaptation and Bioluminescence
> Without ambient light, eyes as we know them are either completely reduced or hyper-developed into [Sensory Array E], capable of detecting even a single stray photon. To interact with their environment, many species produce their own light through a precise reaction between [Reactant 1] and [Catalyst 2] inside small organs called [Luminous Organ Type]. This glow isn't just for sight—it is utilized for [Survival Function Alpha], luring prey via [Behavioral Pattern F], and masking their silhouettes from predators hunting below.
> 
> ### 4. Metabolic Conservation
> Because food resources are sparse and unpredictable, survival requires extreme energy efficiency. Deep-sea life operates in a state of [Metabolic State G], slowing down their [Vital Function H] to near-zero. Many can survive for [Time Duration] between meals. When food does appear—often in the form of [Organic Material Fall] sinking from the upper layers—creatures use specialized appendages like [Morphological Trait I] to capture every single particle.
> 
> Ultimately, these organisms don't just tolerate the deep; they are entirely dependent on it. Take them out of that environment, and the very [Biochemical System J] that keeps them alive would completely destabilize.

Use Gemini API to get embeddings for the baseline answer and compare them with search results

In [46]:
hypothetical_ans = get_embeddings([res.text])

Calculate similarity scores to rank the search results

In [47]:
sim_value = dot_product(search_res, hypothetical_ans)

In [48]:
sim_value

np.float64(0.8649041996670673)

using `np.argmax` best candidate is selected.

**Users's Input:** Explain how deep-sea life survives.

**Answer:**

In [49]:
to_markdown(summaries[np.argmax(sim_value)])

> Based on the provided source, here is the relevant information regarding deep-sea fish adaptations:
> 
> ### Cellular, Biochemical, and Pressure Adaptations
> * **Internal Pressure Balance:** Deep-sea organisms have the same pressure within their bodies as is exerted on them from the outside, preventing them from being crushed.
> * **Membrane Fluidity:** To counteract reduced membrane fluidity caused by extreme hydrostatic pressure, deep-sea cellular membranes favor phospholipid bilayers with an increased proportion of unsaturated fatty acids.
> * **Volume-Decreasing Metabolic Reactions:** High pressure inhibits biochemical reactions that increase in volume and enhances those associated with a decrease in volume; deep-sea fish have evolved metabolic processes and reaction criteria that ultimately decrease system volume.
> * **Enzyme and Protein Resistance:** 
>   * Deep-sea species exhibit lower changes of entropy and enthalpy, favoring negative enthalpy changes and reduced dependence on entropy-driven reactions.
>   * Globular proteins (such as the tertiary structure of G-actin in muscle fibers) are relatively rigid compared to surface fish, structurally strengthened to resist pressure through modification of bonds in their tertiary structure (which also makes them extremely heat-resistant).
>   * The lipoprotein enzyme $\text{Na}^+/\text{K}^+$-ATPase features adaptations in its protein and lipid moieties that give it a much higher tolerance to hydrostatic pressure and compression.
> * **Gelatinous Layer:** About 200 known species (across 10 orders and 13 families) possess a gelatinous layer below the skin or around the spine for buoyancy, low-cost growth, and reducing drag to increase swimming efficiency.
> 
> ### Buoyancy and Body Structure Adaptations
> * **Lack of Swim Bladders / Hydrofoils:** Because high pressure makes maintaining a gas-filled swim bladder energetically costly or prone to compression/expansion risks, swim bladders in deep-sea fishes (particularly bathypelagic species) are usually absent, withered, filled with fat, or scarcely operational. Instead, many exhibit structures similar to hydrofoils to provide hydrodynamic lift.
>   * *(Exceptions: The cusk-eel and rough abyssal grenadier maintain functional swim bladders at extreme depths, allowing them to retain high bone density and strong jaws).*
> * **Reduced Density and Minimal Skeletons:** The deeper a fish lives, the more jelly-like its flesh and the more minimal its bone structure. Tissue density is reduced via high fat content, water accumulation, and reduced skeletal size, thickness, and mineral content.
> * **High Water Content:** Bodies are watery and elongated with weak muscles; having bodies largely composed of water prevents them from being compressed by deep-sea pressures.
> * **Larval and Egg Adaptations:** Eggs and larvae typically drift in the water column and maintain neutral buoyancy using oil droplets in their plasma.
> * **Neoteny:** Bathypelagic fish retain larval-like features (slimy skin without scales; reduced gills, kidneys, hearts, and swim bladders; watery muscles and minimal skeletons) through neoteny, allowing them to stay suspended with minimal energy expenditure.
> * **Locomotion:** Body shapes are typically adapted for periodic bursts of swimming rather than constant swimming.
> 
> ### Vision and Sensory Adaptations
> * **Large, Sensitive Eyes:** Non-blind deep-sea fish possess large, highly sensitive eyes (up to 100 times more sensitive to light than human eyes) adapted to register faint bioluminescence.
> * **Multiple Rh1 Opsin Genes:** While most vertebrates have one Rh1 gene, some deep-sea fish have several, and the silver spinyfin (*Diretmus argenteus*) has 38, aiding vision in dim depths.
> * **Tubular Eyes:** Some deep-water mesopelagic fish have upward-directed tubular eyes with large lenses and exclusively rod cells, providing binocular vision to spot prey silhouetted against ambient light above.
> * **Mirror Optics:** The brownsnout spookfish uses an internal mirror rather than a lens to focus images.
> * **Yellow Lenses:** Some predators possess yellow lenses that filter ambient downwelling light, making prey bioluminescence visible.
> * **Alternative Senses / Blindness:** Many species are blind and rely instead on sensitivities to changes in local pressure (via the lateral line), sound (inner ear), or smell (olfactory systems) to detect food and mates. Some have long feelers to navigate and locate prey.
> 
> ### Bioluminescence and Camouflage
> * **Bioluminescence Production:** Many species possess photophores—light-producing glandular cells containing luminous bacteria bordered by dark pigments, sometimes equipped with lenses to regulate light emanation. Producing light consumes only about 1% of the fish's energy.
> * **Functions of Bioluminescence:** Used to attract prey (e.g., the anglerfish's illuminated lure), find mates, signal territory, communicate, and distract or temporarily blind predators.
> * **Counter-illumination:** Mesopelagic fish illuminate photophores on their bellies to match the color and intensity of downwelling light, eliminating their silhouette from predators lurking below.
> * **Color Camouflage:** 
>   * Ambush predators are colored dark, black, or red (red light wavelengths do not penetrate deep water, making red appear black).
>   * Migratory mesopelagic forms use countershaded silvery coloration.
> 
> ### Feeding and Predation Adaptations
> * **Lures and Feelers:** Deep-sea anglerfish have an illicium (a fishing-rod-like facial adaptation) tipped with a bioluminescent, worm-like piece of skin to lure prey.
> * **Coping with Infrequent Meals:** Due to low food availability, fish adapt to consume prey equal to or larger than themselves by developing:
>   * Disproportionately large mouths
>   * Great, recurved, sharp teeth
>   * Hinged, extensible jaws
>   * Expandable bodies/stomachs
>   * Overlapping gill rakers that keep swallowed prey from escaping.
> * **Low-Energy Foraging:** Bathypelagic species are predominantly sedentary, "sit-and-wait" or "lie-in-wait" ambush predators with slow metabolisms and unspecialized diets to conserve energy.
> 
> ### Vertical Migration and Temperature Adaptations
> * **Daily Vertical Migrations:** Many mesopelagic fish migrate up to the epipelagic zone at dusk to feed on plankton and descend at dawn. They use swim bladders (inflating them to ascend and deflating them to descend).
> * **Thermal Tolerance:** Migrating through ocean thermoclines exposes mesopelagic fish to temperature shifts between 10 °C and 21 °C (50–69 °F), requiring wide temperature tolerances.
> 
> ### Reproductive Adaptations
> * **Finding Mates in Darkness:** Species use species-specific bioluminescent patterns, release of pheromones, and highly developed olfactory systems (allowing males to track females by scent).
> * **Hermaphroditism:** Some species are hermaphrodites, ensuring any encounter between two individuals can result in egg and sperm production.
> * **Parasitic Mating (Sexual Dimorphism):** In certain anglerfishes (e.g., *Haplophryne mollis*), a tiny male bites the female and releases an enzyme that dissolves the tissue of his mouth and her skin, permanently fusing their circulatory systems. The male atrophies into a sperm-producing organ attached to the female, ensuring an immediate mate when she is ready to spawn.
> 
> Based on:
>   https://en.wikipedia.org/wiki/Deep-sea_fish

You have now created a search re-ranking engine using embeddings!

## Next steps

I hope you found this example helpful! Check out more examples in the [Gemini API cookbook](https://github.com/google-gemini/cookbook/) to learn more.